In [ ]:

# Training ResNet50V2 y metrics

from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import os

# Directories of Plantar Thermograms for Training and Validation
train_dir = r'C:\Users\Marben\Desktop\Modelo MovileNet2 con RB\Cluster\MobileNetV2RbBl\Train'
val_dir   = r'C:\Users\Marben\Desktop\Modelo MovileNet2 con RB\Cluster\MobileNetV2RbBl\Val'

# Image Augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1.0/255.0)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

validation_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

print("\n📊 Información del conjunto de datos:")
print(f" - Imágenes de entrenamiento: {train_generator.samples}")
print(f" - Imágenes de validación: {validation_generator.samples}")

# Load ResNet50V2 base (congelada) ===
base_model = ResNet50V2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_model.layers:
    layer.trainable = False  

# Custom head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
output = Dense(train_generator.num_classes, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=output)

# Compilation of model
model.compile(optimizer=Adam(1e-4),  # puedes usar 1e-4 o 1e-3 si no hay fine-tuning
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Training
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=300
)

# Plots
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Precisión (Accuracy)')
plt.xlabel('Época')
plt.ylabel('Precisión')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Pérdida (Loss)')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Model evaluation 
print('🔎 Evaluando modelo con datos de validación...')
Y_pred = model.predict(validation_generator)
y_pred = np.argmax(Y_pred, axis=1)
y_true = validation_generator.classes
class_labels = list(validation_generator.class_indices.keys())

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.title('Matriz de Confusión')
plt.show()

# Classification report
print('📋 Reporte de Clasificación:')
print(classification_report(y_true, y_pred, target_names=class_labels))

# Additional metrics
print("\n📌 Métricas Adicionales:")
print(f"Exactitud (Accuracy): {accuracy_score(y_true, y_pred):.4f}")
print(f"Precisión (Precision): {precision_score(y_true, y_pred, average='macro'):.4f}")
print(f"Sensibilidad (Recall): {recall_score(y_true, y_pred, average='macro'):.4f}")
print(f"F1-Score (macro promedio): {f1_score(y_true, y_pred, average='macro'):.4f}")

# Specificity per class
print("\n📌 Especificidad por clase:")
for i, label in enumerate(class_labels):
    true_neg = np.sum(np.delete(np.delete(cm, i, axis=0), i, axis=1))
    false_pos = np.sum(cm[:, i]) - cm[i, i]
    specificity = true_neg / (true_neg + false_pos) if (true_neg + false_pos) > 0 else 0
    print(f" - {label}: {specificity:.4f}")

# === Save model
model.save('modelo_resnet50v2_sin_finetuning_rb_bl.keras')


In [ ]:
# Training VGG19 y metrics

from tensorflow.keras.applications import VGG19
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import os

# Directories of Plantar Thermograms for Training and Validation
train_dir = r'C:\Users\Marben\Desktop\Modelo MovileNet2 con RB\Cluster\MobileNetV2RbBl\Train'
val_dir   = r'C:\Users\Marben\Desktop\Modelo MovileNet2 con RB\Cluster\MobileNetV2RbBl\Val'

# Image augmentation 
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1.0/255.0)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

validation_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

# Display Dataset Information
print("\n📊 Información del conjunto de datos:")
print(f" - Imágenes de entrenamiento: {train_generator.samples}")
print(f" - Imágenes de validación: {validation_generator.samples}")

# Load VGG19 base
base_model = VGG19(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze the Last 50 Layers
for layer in base_model.layers[-42:]:
    layer.trainable = True

# Final Model Construction 
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
output = Dense(train_generator.num_classes, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=output)

# Compilation of Model 
model.compile(optimizer=Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

# Training
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs= 60
)

# Plots
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Precisión (Accuracy)')
plt.xlabel('Época')
plt.ylabel('Precisión')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Pérdida (Loss)')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Final Evaluation 
print('🔎 Evaluando modelo con datos de validación...')
Y_pred = model.predict(validation_generator)
y_pred = np.argmax(Y_pred, axis=1)
y_true = validation_generator.classes
class_labels = list(validation_generator.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.title('Matriz de Confusión')
plt.show()

print('📋 Reporte de Clasificación:')
print(classification_report(y_true, y_pred, target_names=class_labels))

print("\n📌 Métricas Adicionales:")
print(f"Exactitud (Accuracy): {accuracy_score(y_true, y_pred):.4f}")
print(f"Precisión (Precision): {precision_score(y_true, y_pred, average='macro'):.4f}")
print(f"Sensibilidad (Recall): {recall_score(y_true, y_pred, average='macro'):.4f}")
print(f"F1-Score (macro promedio): {f1_score(y_true, y_pred, average='macro'):.4f}")

print("\n📌 Especificidad por clase:")
for i, label in enumerate(class_labels):
    true_neg = np.sum(np.delete(np.delete(cm, i, axis=0), i, axis=1))
    false_pos = np.sum(cm[:, i]) - cm[i, i]
    specificity = true_neg / (true_neg + false_pos) if (true_neg + false_pos) > 0 else 0
    print(f" - {label}: {specificity:.4f}")

# Save model
model.save('modelo_vgg19_finetuned_rb_bl.keras')


In [ ]:
# Training MobileNetV2
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

print('Iniciando el proceso...')


# Data preparation
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2  # Reservar un 20% para validación
)

# Training and validation generator 
train_generator = train_datagen.flow_from_directory(
    r'C:/Users/Marben/Desktop/Modelo MovileNet2 con RB/Training',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

validation_generator = train_datagen.flow_from_directory(
    r'C:/Users/Marben/Desktop/Modelo MovileNet2 con RB/Training',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# =======================
# 2️⃣ Modelo Base (MobileNetV2)
# =======================
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# add layers custom
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# Congelar las capas del modelo base
for layer in base_model.layers:
    layer.trainable = False


#  Model compilation 

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Early Stopping para evitar sobreajuste
early_stop = EarlyStopping(monitor='val_loss', patience= 15, restore_best_weights=True)


# Model tryning
print('Iniciando el entrenamiento...')
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=90,
    callbacks=[early_stop],
    verbose=1  # Muestra el progreso por época
)


# Model evaluation

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Precisión durante el Entrenamiento')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Pérdida durante el Entrenamiento')
plt.legend()
plt.grid(True)

plt.show()


# Optimized prediction

print('Generando matriz de confusión y reporte de clasificación...')

Y_pred = model.predict(validation_generator, steps=len(validation_generator), verbose=1)
y_pred = np.argmax(Y_pred, axis=1)


# Confusion matrix

plt.figure(figsize=(8, 6))
cm = confusion_matrix(validation_generator.classes, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=list(train_generator.class_indices.keys()), 
            yticklabels=list(train_generator.class_indices.keys()))
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.title('Matriz de Confusión')
plt.show()

print('Reporte de Clasificación:')
print(classification_report(validation_generator.classes, y_pred, 
                            target_names=list(train_generator.class_indices.keys())))

# Model Save

model.save('modelo_mobilenetv2_4clases.keras')
print('Modelo guardado exitosamente en modelo_mobilenetv2_4clases.keras')

